In [24]:
# -------------------------------------------------
# PROJECT SETUP
# -------------------------------------------------

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\dev\util


In [25]:
# -------------------------------------------------
# IMPORT UTIL MODULES
# -------------------------------------------------

from src.data_fetcher import build_forecast_table
from src.baseline import build_baseline_schedule
from src.optimizer import optimize_schedule
from src.metrics import compare_schedules

In [26]:
# -------------------------------------------------
# LOAD PLACEHOLDER FORECAST DATA
# -------------------------------------------------

forecast_df = build_forecast_table(carbon_path, price_path)

print("Rows in forecast:", len(forecast_df))
forecast_df.head(10)

Rows in forecast: 24


,timestamp,carbon_g_per_kwh,price_per_kwh
0,2026-03-13 00:00:00,340,0.09
1,2026-03-13 01:00:00,330,0.08
2,2026-03-13 02:00:00,320,0.08
3,2026-03-13 03:00:00,310,0.07
4,2026-03-13 04:00:00,300,0.07
5,2026-03-13 05:00:00,290,0.08
6,2026-03-13 06:00:00,280,0.10
7,2026-03-13 07:00:00,270,0.13
8,2026-03-13 08:00:00,220,0.16
9,2026-03-13 09:00:00,210,0.18


In [27]:
baseline_df = build_baseline_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8
)

baseline_df

,timestamp,carbon_g_per_kwh,price_per_kwh,baseline_run_flag
0,2026-03-13 00:00:00,340,0.09,1
1,2026-03-13 01:00:00,330,0.08,1
2,2026-03-13 02:00:00,320,0.08,1
3,2026-03-13 03:00:00,310,0.07,1
4,2026-03-13 04:00:00,300,0.07,1
5,2026-03-13 05:00:00,290,0.08,1
6,2026-03-13 06:00:00,280,0.10,1
7,2026-03-13 07:00:00,270,0.13,1
8,2026-03-13 08:00:00,220,0.16,0
9,2026-03-13 09:00:00,210,0.18,0


In [28]:
baseline_df = build_baseline_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8
)

optimized_df = optimize_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8,
    objective="carbon"
)

optimized_df

,timestamp,carbon_g_per_kwh,price_per_kwh,score,run_flag
0,2026-03-13 00:00:00,340,0.09,340,0
1,2026-03-13 01:00:00,330,0.08,330,0
2,2026-03-13 02:00:00,320,0.08,320,0
3,2026-03-13 03:00:00,310,0.07,310,0
4,2026-03-13 04:00:00,300,0.07,300,0
5,2026-03-13 05:00:00,290,0.08,290,0
6,2026-03-13 06:00:00,280,0.10,280,0
7,2026-03-13 07:00:00,270,0.13,270,0
8,2026-03-13 08:00:00,220,0.16,220,0
9,2026-03-13 09:00:00,210,0.18,210,0


In [29]:
comparison = compare_schedules(
    baseline_df=baseline_df,
    optimized_df=optimized_df,
    machine_watts=300
)

import pandas as pd
comparison_df = pd.DataFrame([comparison])

comparison_df

,baseline_cost,optimized_cost,cost_savings,cost_reduction_pct,baseline_carbon_kg,optimized_carbon_kg,carbon_savings_kg,carbon_reduction_pct
0,0.21,0.678,-0.468,-222.857143,0.732,0.378,0.354,48.360656


In [30]:
optimized_df[optimized_df["run_flag"] == 1][["timestamp","carbon_g_per_kwh"]]

,timestamp,carbon_g_per_kwh
10,2026-03-13 10:00:00,200
11,2026-03-13 11:00:00,180
12,2026-03-13 12:00:00,160
13,2026-03-13 13:00:00,150
14,2026-03-13 14:00:00,140
15,2026-03-13 15:00:00,130
16,2026-03-13 16:00:00,140
17,2026-03-13 17:00:00,160


In [31]:
baseline_df[baseline_df["baseline_run_flag"] == 1][["timestamp","carbon_g_per_kwh"]]

,timestamp,carbon_g_per_kwh
0,2026-03-13 00:00:00,340
1,2026-03-13 01:00:00,330
2,2026-03-13 02:00:00,320
3,2026-03-13 03:00:00,310
4,2026-03-13 04:00:00,300
5,2026-03-13 05:00:00,290
6,2026-03-13 06:00:00,280
7,2026-03-13 07:00:00,270


In [32]:
optimized_df[optimized_df["run_flag"] == 1][["timestamp","carbon_g_per_kwh"]]

,timestamp,carbon_g_per_kwh
10,2026-03-13 10:00:00,200
11,2026-03-13 11:00:00,180
12,2026-03-13 12:00:00,160
13,2026-03-13 13:00:00,150
14,2026-03-13 14:00:00,140
15,2026-03-13 15:00:00,130
16,2026-03-13 16:00:00,140
17,2026-03-13 17:00:00,160


In [33]:
comparison = compare_schedules(
    baseline_df=baseline_df,
    optimized_df=optimized_df,
    machine_watts=300
)

import pandas as pd
pd.DataFrame([comparison])

,baseline_cost,optimized_cost,cost_savings,cost_reduction_pct,baseline_carbon_kg,optimized_carbon_kg,carbon_savings_kg,carbon_reduction_pct
0,0.21,0.678,-0.468,-222.857143,0.732,0.378,0.354,48.360656


In [35]:
optimized_deadline_df = optimize_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8,
    objective="carbon",
    deadline="2026-03-13 17:00"
)

optimized_deadline_df[optimized_deadline_df["run_flag"] == 1][
    ["timestamp", "carbon_g_per_kwh", "eligible_flag", "run_flag"]
]

,timestamp,carbon_g_per_kwh,eligible_flag,run_flag
10,2026-03-13 10:00:00,200,1,1
11,2026-03-13 11:00:00,180,1,1
12,2026-03-13 12:00:00,160,1,1
13,2026-03-13 13:00:00,150,1,1
14,2026-03-13 14:00:00,140,1,1
15,2026-03-13 15:00:00,130,1,1
16,2026-03-13 16:00:00,140,1,1
17,2026-03-13 17:00:00,160,1,1


In [36]:
optimized_tight_df = optimize_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8,
    objective="carbon",
    deadline="2026-03-13 12:00"
)

optimized_tight_df[optimized_tight_df["run_flag"] == 1][
    ["timestamp", "carbon_g_per_kwh", "eligible_flag", "run_flag"]
]

,timestamp,carbon_g_per_kwh,eligible_flag,run_flag
5,2026-03-13 05:00:00,290,1,1
6,2026-03-13 06:00:00,280,1,1
7,2026-03-13 07:00:00,270,1,1
8,2026-03-13 08:00:00,220,1,1
9,2026-03-13 09:00:00,210,1,1
10,2026-03-13 10:00:00,200,1,1
11,2026-03-13 11:00:00,180,1,1
12,2026-03-13 12:00:00,160,1,1


In [37]:
comparison_deadline = compare_schedules(
    baseline_df=baseline_df,
    optimized_df=optimized_deadline_df,
    machine_watts=300
)

import pandas as pd
pd.DataFrame([comparison_deadline])

,baseline_cost,optimized_cost,cost_savings,cost_reduction_pct,baseline_carbon_kg,optimized_carbon_kg,carbon_savings_kg,carbon_reduction_pct
0,0.21,0.678,-0.468,-222.857143,0.732,0.378,0.354,48.360656


In [38]:
from src.scheduler import format_schedule

formatted_schedule_df = format_schedule(optimized_deadline_df)

formatted_schedule_df

,timestamp,eligible_flag,run_flag,recommended_action,price_per_kwh,carbon_g_per_kwh
0,2026-03-13 00:00:00,1,0,Wait,0.09,340
1,2026-03-13 01:00:00,1,0,Wait,0.08,330
2,2026-03-13 02:00:00,1,0,Wait,0.08,320
3,2026-03-13 03:00:00,1,0,Wait,0.07,310
4,2026-03-13 04:00:00,1,0,Wait,0.07,300
5,2026-03-13 05:00:00,1,0,Wait,0.08,290
6,2026-03-13 06:00:00,1,0,Wait,0.10,280
7,2026-03-13 07:00:00,1,0,Wait,0.13,270
8,2026-03-13 08:00:00,1,0,Wait,0.16,220
9,2026-03-13 09:00:00,1,0,Wait,0.18,210


In [39]:
formatted_schedule_df[formatted_schedule_df["recommended_action"] == "Run"]

,timestamp,eligible_flag,run_flag,recommended_action,price_per_kwh,carbon_g_per_kwh
10,2026-03-13 10:00:00,1,1,Run,0.20,200
11,2026-03-13 11:00:00,1,1,Run,0.22,180
12,2026-03-13 12:00:00,1,1,Run,0.25,160
13,2026-03-13 13:00:00,1,1,Run,0.27,150
14,2026-03-13 14:00:00,1,1,Run,0.30,140
15,2026-03-13 15:00:00,1,1,Run,0.32,130
16,2026-03-13 16:00:00,1,1,Run,0.34,140
17,2026-03-13 17:00:00,1,1,Run,0.36,160


In [40]:
from src.inputs import WorkloadInput

sample_input = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-13 17:00",
    objective="carbon",
    machine_watts=300,
)

sample_input

WorkloadInput(zip_code='93106', compute_hours_required=8, deadline='2026-03-13 17:00', objective='carbon', machine_watts=300)

In [41]:
bad_input = WorkloadInput(
    zip_code="9310",
    compute_hours_required=8,
    deadline="2026-03-13 17:00",
    objective="carbon",
    machine_watts=300,
)

ValueError: zip_code must be a 5-digit string